In [5]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import count

In [6]:
spark = SparkSession.builder.appName("ParquetDemo").master("local[*]").getOrCreate()

In [7]:
data = [ ("Targaryen",), ("Targaryen",), ("Targaryen",), ("Stark",), ("Stark",), ("Baratheon",) ]

In [8]:
df = spark.createDataFrame(data, ["allegiance"])
print("Original Data") 
df.show()

Original Data
+----------+
|allegiance|
+----------+
| Targaryen|
| Targaryen|
| Targaryen|
|     Stark|
|     Stark|
| Baratheon|
+----------+



In [9]:
df_final_counts = ( df.groupBy("allegiance") .agg(count("*").alias("total_count")) )
print("Aggregated Data") 
df_final_counts.show()

Aggregated Data
+----------+-----------+
|allegiance|total_count|
+----------+-----------+
| Targaryen|          3|
|     Stark|          2|
| Baratheon|          1|
+----------+-----------+



In [10]:
parquet_path = "./output/parquet_records" 
df_final_counts.write.mode("overwrite").parquet(parquet_path) 
print(f"Written to: {parquet_path}")

Written to: ./output/parquet_records


In [11]:
df_history = spark.read.parquet(parquet_path) 
print("Read From Parquet") 
df_history.show() 
print("Schema Inferred Automatically") 
df_history.printSchema() 
spark.stop()

Read From Parquet
+----------+-----------+
|allegiance|total_count|
+----------+-----------+
| Targaryen|          3|
|     Stark|          2|
| Baratheon|          1|
+----------+-----------+

Schema Inferred Automatically
root
 |-- allegiance: string (nullable = true)
 |-- total_count: long (nullable = true)



## 1. Data I/O: Parquet and Delta Lake

When building backend applications, structured and transactional data is typically stored in databases such as PostgreSQL. In the big data ecosystem, however, data is commonly stored in distributed file systems like **Amazon S3** or **HDFS** using highly optimized storage formats.

---

### Parquet

**Parquet** is a columnar storage format designed for analytical workloads.

Unlike traditional row-based storage, Parquet stores data column by column, enabling Spark to read only the columns required for a query.

### Key Benefits

- Columnar storage
- High compression ratios
- Built-in schema storage
- Faster analytical queries
- Reduced disk and network I/O

### Example

Suppose a dataset contains 100 columns:

```python
df.select("knight_name")
```

With Parquet, Spark physically reads only the `knight_name` column from disk rather than scanning all 100 columns.

```text
Parquet File
├── knight_name      ✓ Read
├── house            ✗ Skip
├── age              ✗ Skip
├── rank             ✗ Skip
└── ...              ✗ Skip
```

This significantly reduces:

- Disk reads
- Network traffic
- Memory usage
- Query execution time

---

### Delta Lake

**Delta Lake** is an open-source storage layer built on top of Parquet.

While Parquet files are immutable, Delta Lake adds a transaction log that enables database-like functionality on large-scale datasets.

### What Delta Lake Adds

- ACID transactions
- Data versioning
- Schema enforcement
- Schema evolution
- Efficient updates and deletes
- Time travel

### Why It Exists

With plain Parquet, operations such as:

```sql
UPDATE records
SET rank = 'Kingsguard'
WHERE knight_name = 'Dunk';
```

are not possible directly because Parquet files cannot be modified in place.

Delta Lake solves this by maintaining a transaction log that tracks changes and versions of the underlying Parquet files.

---

### Time Travel

One of Delta Lake's most powerful features is **Time Travel**.

You can query previous versions of a dataset:

```python
spark.read.format("delta") \
    .option("versionAsOf", 5) \
    .load("/data/knights")
```

This allows you to:

- Audit historical data
- Recover from accidental updates
- Reproduce previous analyses
- Compare dataset versions

---

### Parquet vs Delta Lake

| Feature | Parquet | Delta Lake |
|----------|----------|------------|
| Columnar Storage | ✅ Yes | ✅ Yes |
| Compression | ✅ Yes | ✅ Yes |
| Schema Storage | ✅ Yes | ✅ Yes |
| ACID Transactions | ❌ No | ✅ Yes |
| UPDATE / DELETE Support | ❌ No | ✅ Yes |
| Time Travel | ❌ No | ✅ Yes |
| Transaction Log | ❌ No | ✅ Yes |
| Schema Enforcement | ❌ Limited | ✅ Yes |

---

### Rule of Thumb

> Use **Parquet** when you need efficient, compressed, columnar storage for analytical workloads.
>
> Use **Delta Lake** when you need database-like capabilities such as transactions, updates, deletes, schema management, and historical version tracking on large-scale datasets.